# Flipkart Data Pipeline - Medallion Architecture

## Pipeline Overview
**Bronze Layer** → Raw data ingestion  
**Silver Layer** → Data cleaning & transformation  
**Gold Layer** → Business aggregations & insights

## Business Requirements
1. **Sales Analysis**: Total sales by product, category, city
2. **Customer Insights**: Orders per customer, top customers
3. **Data Quality**: No NULLs, duplicates, or negative values
4. **Latest Data**: Keep most recent record for duplicate order_ids
5. **Dashboard Ready**: Clean, aggregated data for BI tools

In [0]:
# Load raw data as-is to Bronze layer (includes nulls, duplicates, invalid values)

from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

data = [
    (101, "C001", "Laptop", "Electronics", "Hyderabad", "2024-01-01", "50000", 1),
    (102, "C002", "Mobile", "Electronics", "Chennai", "2024-01-02", None, 2),
    (103, "C001", "Tablet", "Electronics", "Hyderabad", "2024-01-03", "20000", 1),
    (104, "C003", "Laptop", "Electronics", "Delhi", "2024-01-04", "55000", 1),
    (105, "C002", "Mobile", "Electronics", "Chennai", "2024-01-05", "18000", 1),
    (106, "C004", "Watch", "Accessories", "Mumbai", "2024-01-06", "8000", 1),
    (103, "C001", "Tablet", "Electronics", "Hyderabad", "2024-01-07", "22000", 1),
    (107, "C005", "Headphones", "Accessories", None, "2024-01-08", "3000", 1),
    (108, "C006", "Laptop", "Electronics", "Bangalore", "2024-01-09", "-45000", 1),
    (109, "C007", "Mobile", "Electronics", "Chennai", "2024-01-10", "15000", 2),
    (109, "C007", "Mobile", "Electronics", "Chennai", "2024-01-10", "15000", 2)
]

columns = ["order_id", "customer_id", "product", "category", "city", "date", "amount", "quantity"]

df_bronze = spark.createDataFrame(data, columns)
bronze_path = "/Volumes/workspace/default/medallionarchitecture/bronze/flipkart_orders"
df_bronze.write.format("delta").mode("overwrite").save(bronze_path)

print("✓ Bronze Layer Created")
print(f"Location: {bronze_path}")
print(f"Total Records: {df_bronze.count()}")
display(df_bronze)

✓ Bronze Layer Created
Location: /Volumes/workspace/default/medallionarchitecture/bronze/flipkart_orders
Total Records: 11


order_id,customer_id,product,category,city,date,amount,quantity
101,C001,Laptop,Electronics,Hyderabad,2024-01-01,50000,1
102,C002,Mobile,Electronics,Chennai,2024-01-02,null,2
103,C001,Tablet,Electronics,Hyderabad,2024-01-03,20000,1
104,C003,Laptop,Electronics,Delhi,2024-01-04,55000,1
105,C002,Mobile,Electronics,Chennai,2024-01-05,18000,1
106,C004,Watch,Accessories,Mumbai,2024-01-06,8000,1
103,C001,Tablet,Electronics,Hyderabad,2024-01-07,22000,1
107,C005,Headphones,Accessories,null,2024-01-08,3000,1
108,C006,Laptop,Electronics,Bangalore,2024-01-09,-45000,1
109,C007,Mobile,Electronics,Chennai,2024-01-10,15000,2


In [0]:
# Read Bronze data and fill NULL values with defaults

df = spark.read.format("delta").load(bronze_path)
df_clean = df.fillna({"amount": "1000", "city": "Unknown"})
display(df_clean)

order_id,customer_id,product,category,city,date,amount,quantity
101,C001,Laptop,Electronics,Hyderabad,2024-01-01,50000,1
102,C002,Mobile,Electronics,Chennai,2024-01-02,1000,2
103,C001,Tablet,Electronics,Hyderabad,2024-01-03,20000,1
104,C003,Laptop,Electronics,Delhi,2024-01-04,55000,1
105,C002,Mobile,Electronics,Chennai,2024-01-05,18000,1
106,C004,Watch,Accessories,Mumbai,2024-01-06,8000,1
103,C001,Tablet,Electronics,Hyderabad,2024-01-07,22000,1
107,C005,Headphones,Accessories,Unknown,2024-01-08,3000,1
108,C006,Laptop,Electronics,Bangalore,2024-01-09,-45000,1
109,C007,Mobile,Electronics,Chennai,2024-01-10,15000,2


In [0]:
# Convert amount to integer and date to date type

from pyspark.sql.functions import *
df_clean = df_clean.withColumn("amount", col("amount").cast("int")).withColumn("date", to_date(col("date"),"yyyy-MM-dd"))

In [0]:
# Filter out negative amounts

df_filtered = df_clean.filter(col("amount") > 0)

In [0]:
# Keep only the latest record for duplicate order_ids

from pyspark.sql.window import Window
w = Window.partitionBy("order_id").orderBy(col("date").desc())
df_filtered = df_filtered.withColumn("row_num", row_number().over(w)).filter(col("row_num") == 1).drop("row_num")

In [0]:
# Save cleaned data to Silver layer

silver_path = "/Volumes/workspace/default/medallionarchitecture/silver/flipkart_orders"
df_filtered.write.format("delta").mode("overwrite").save(silver_path)
df_silver = spark.read.format("delta").load(silver_path)

In [0]:
# Save Silver data to Gold layer for analytics

gold_path = "/Volumes/workspace/default/medallionarchitecture/gold/flipkart_orders"
df_silver.write.format("delta").mode("overwrite").save(gold_path)

In [0]:
# Read Gold layer data

df_gold = spark.read.format("delta").load(gold_path)
display(df_gold)

order_id,customer_id,product,category,city,date,amount,quantity
101,C001,Laptop,Electronics,Hyderabad,2024-01-01,50000,1
102,C002,Mobile,Electronics,Chennai,2024-01-02,1000,2
103,C001,Tablet,Electronics,Hyderabad,2024-01-07,22000,1
104,C003,Laptop,Electronics,Delhi,2024-01-04,55000,1
105,C002,Mobile,Electronics,Chennai,2024-01-05,18000,1
106,C004,Watch,Accessories,Mumbai,2024-01-06,8000,1
107,C005,Headphones,Accessories,Unknown,2024-01-08,3000,1
109,C007,Mobile,Electronics,Chennai,2024-01-10,15000,2


In [0]:
# Calculate total sales by product

df_product = df_gold.groupBy("product").agg(sum("amount").alias("total_sales"))

In [0]:
# Calculate total sales by category

df_category = df_gold.groupBy("category").agg(sum("amount").alias("total_sales"))

In [0]:
# Calculate total sales by city

df_city = df_gold.groupBy("city").agg(sum("amount").alias("total_sales"))

In [0]:
# Calculate total orders and spending per customer

df_customer = df_gold.groupBy("customer_id").agg(count("order_id").alias("total_orders"), sum("amount").alias("total_spent"))

In [0]:
# Find the customer who spent the most

df_customer.orderBy(col("total_spent").desc()).limit(1).show()

+-----------+------------+-----------+
|customer_id|total_orders|total_spent|
+-----------+------------+-----------+
|       C001|           2|      72000|
+-----------+------------+-----------+



In [0]:
# Find the product with highest total sales

df_product.orderBy(col("total_sales").desc()).limit(1).show()

+-------+-----------+
|product|total_sales|
+-------+-----------+
| Laptop|     105000|
+-------+-----------+

